## Woolworths の緯度・経度を取得する

Coles と同様に Overpass API を用いてオーストラリア全土の Woolworths 店舗座標を取得し、
`notebook/output/woolworths_locations.csv` に保存する。

このデータは `coles_correlation.ipynb` での交差相関関数 $\xi_{CW}(r)$ の計算に使用する。

In [7]:
%use dataframe
%use ktor-client

In [8]:
// const
object DirectoryPath {
    const val output = "./output/"
}

## Overpass API でオーストラリア全土の Woolworths 店舗を取得する

OSM タグ: `shop=supermarket` + `brand=Woolworths`

Note: Woolworths Metro（小型フォーマット）も `brand=Woolworths` でタグ付けされている場合があるため、
取得後に件数を確認すること（Coles ~685 点に対して Woolworths は ~1,000–1,100 点程度が期待値）。

In [9]:
import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.*
import io.ktor.client.request.*
import io.ktor.client.statement.*
import io.ktor.http.*
import kotlinx.coroutines.runBlocking

val http = HttpClient(CIO) {
    install(HttpTimeout) {
        requestTimeoutMillis = 300_000
        connectTimeoutMillis = 60_000
        socketTimeoutMillis = 300_000
    }
}

val areaId = "3600080500" // オーストラリア全土
val brand = "Woolworths"

fun buildOverpassQuery(areaId: String, brand: String): String = """
   [out:json][timeout:300];
   area(${areaId})->.searchArea;
   nwr["shop"="supermarket"]["brand"="$brand"](area.searchArea);
   out center qt;
""".trimIndent()

val result = runBlocking {
    val query = buildOverpassQuery(areaId, brand)

    val response = http.post("https://overpass-api.de/api/interpreter") {
        contentType(ContentType.Application.FormUrlEncoded)
        setBody("data=" + query)
    }
    response.bodyAsText()
}

## JSON パース → Data Class リスト化

Coles と同じフィールド構成: `id`, `lat`, `lon`, `state`, `suburb`, `street`

In [10]:
import kotlinx.serialization.json.*

data class WoolworthsLocation(
    val id: Long,
    val lat: Double,
    val lon: Double,
    val state: String? = null,
    val suburb: String? = null,
    val street: String? = null
)

val root = Json.parseToJsonElement(result).jsonObject
val elements = root["elements"]!!.jsonArray

val locations = elements.mapNotNull { element ->
    val obj = element.jsonObject

    val id  = obj["id"]?.jsonPrimitive?.longOrNull
    val lat = obj["lat"]?.jsonPrimitive?.doubleOrNull
    val lon = obj["lon"]?.jsonPrimitive?.doubleOrNull

    // way/relation 要素は "center" キー内に緯度経度を持つ
    val centerLat = obj["center"]?.jsonObject?.get("lat")?.jsonPrimitive?.doubleOrNull
    val centerLon = obj["center"]?.jsonObject?.get("lon")?.jsonPrimitive?.doubleOrNull

    val finalLat = lat ?: centerLat
    val finalLon = lon ?: centerLon

    if (id == null || finalLat == null || finalLon == null) return@mapNotNull null

    val tags   = obj["tags"]?.jsonObject
    val state  = tags?.get("addr:state")?.jsonPrimitive?.contentOrNull
    val suburb = tags?.get("addr:suburb")?.jsonPrimitive?.contentOrNull
    val street = tags?.get("addr:street")?.jsonPrimitive?.contentOrNull

    WoolworthsLocation(id = id, lat = finalLat, lon = finalLon,
                       state = state, suburb = suburb, street = street)
}

println("Woolworths 店舗数: ${locations.size}")

kotlinx.serialization.json.internal.JsonDecodingException: Unexpected JSON token at offset 7: Expected EOF after parsing, but had v instead at path: $
JSON input: <?xml version="1.0" encoding="UTF-8"?.....

## CSV に保存する

出力先: `notebook/output/woolworths_locations.csv`

In [5]:
import java.io.File

fun exportToCsv(locations: List<WoolworthsLocation>, filePath: String) {
    val file = File(filePath)
    file.bufferedWriter().use { writer ->
        writer.appendLine("id,lat,lon,state,suburb,street")
        for (loc in locations) {
            val line = listOf(
                loc.id.toString(),
                loc.lat.toString(),
                loc.lon.toString(),
                loc.state  ?: "",
                loc.suburb ?: "",
                loc.street ?: ""
            ).joinToString(",")
            writer.appendLine(line)
        }
    }
    println("保存完了: $filePath  (${locations.size} 行)")
}

exportToCsv(locations, DirectoryPath.output + "woolworths_locations.csv")

保存完了: ./output/woolworths_locations.csv  (1039 行)


## 簡易 QC: 州別件数・緯度経度範囲

Woolworths は 2024 年時点でオーストラリア全土に約 1,050–1,100 店舗を持つ。
件数が大きくずれる場合は OSM の `brand` タグが `Woolworths Supermarkets` 等になっている可能性がある。

In [6]:
// 州別件数
val byState = locations.groupBy { it.state ?: "(unknown)" }
    .map { (state, locs) -> state to locs.size }
    .sortedByDescending { it.second }

println("--- 州別件数 ---")
byState.forEach { (state, count) -> println("  $state: $count") }

// 緯度・経度範囲
val lats = locations.map { it.lat }
val lons = locations.map { it.lon }
println("\n--- 緯度・経度範囲 ---")
println("  lat: ${lats.min()} 〜 ${lats.max()}")
println("  lon: ${lons.min()} 〜 ${lons.max()}")

--- 州別件数 ---
  (unknown): 869
  WA: 65
  NSW: 38
  VIC: 35
  QLD: 16
  TAS: 9
  SA: 6
  NT: 1

--- 緯度・経度範囲 ---
  lat: -43.0279391 〜 -12.183894
  lon: 113.6576857 〜 153.6120783
